In [1]:
!pip install pyspark

In [2]:
import os
import glob
import platform
# --- Ensure a JVM is available for PySpark (Spark 4 needs Java 17+) ---
# Auto-detect a local JDK on macOS, Linux OR Windows if JAVA_HOME isn't already set.
def _find_java_home():
    jh = os.environ.get("JAVA_HOME")
    if jh and os.path.isdir(jh):
        return jh
    system = platform.system()
    candidates = []
    if system == "Darwin":            # macOS (Homebrew)
        candidates += ["/opt/homebrew/opt/openjdk@17", "/opt/homebrew/opt/openjdk@21",
                       "/opt/homebrew/opt/openjdk", "/usr/local/opt/openjdk@17"]
    elif system == "Windows":         # common Windows JDK install locations
        for base in (r"C:\Program Files\Eclipse Adoptium",
                     r"C:\Program Files\Java",
                     r"C:\Program Files\Microsoft\jdk-17",
                     r"C:\Program Files\Amazon Corretto",
                     r"C:\Program Files\Zulu"):
            candidates += sorted(glob.glob(base + r"*17*"))
            candidates += sorted(glob.glob(base + r"*21*"))
            candidates += sorted(glob.glob(base + r"*"))
    else:                             # Linux
        for pat in ("*17*", "*21*", "*"):
            candidates += sorted(glob.glob(f"/usr/lib/jvm/{pat}"))
    exe = "java.exe" if system == "Windows" else "java"
    for c in candidates:
        if os.path.isfile(os.path.join(c, "bin", exe)):
            return c
    return None

_jh = _find_java_home()
if _jh:
    os.environ["JAVA_HOME"] = _jh
    os.environ["PATH"] = os.path.join(_jh, "bin") + os.pathsep + os.environ["PATH"]
    print("JAVA_HOME =", _jh)
else:
    print("No JDK found automatically — install Java 17+ and set JAVA_HOME manually.")

from pyspark.sql import SparkSession

spark = (
    SparkSession.builder
    .appName("read-netflix-sources")
    # Force Spark onto loopback so it doesn't bind to a LAN/VPN address
    # (avoids "Connection reset" when initialising the local driver on macOS).
    .config("spark.driver.bindAddress", "127.0.0.1")
    .config("spark.driver.host", "127.0.0.1")
    # MySQL JDBC driver on the classpath (downloaded from Maven on first run).
    # This must be set BEFORE the session starts — restart the kernel if you change it.
    .config("spark.jars.packages", "com.mysql:mysql-connector-j:9.1.0")
    .getOrCreate()
)
spark

JAVA_HOME = /usr/lib/jvm/java-1.17.0-openjdk-amd64


In [3]:
CSV_PATH = "/content/netflix_titles.csv"

raw = (
    spark.read
    .option("header", True)
    .option("inferSchema", True)
    .csv(CSV_PATH)
)

print("Dataset loaded successfully")
print("Total rows:", raw.count())
print("Total columns:", len(raw.columns))

Dataset loaded successfully
Total rows: 8809
Total columns: 12


In [4]:
raw.printSchema()

root
 |-- show_id: string (nullable = true)
 |-- type: string (nullable = true)
 |-- title: string (nullable = true)
 |-- director: string (nullable = true)
 |-- cast: string (nullable = true)
 |-- country: string (nullable = true)
 |-- date_added: string (nullable = true)
 |-- release_year: string (nullable = true)
 |-- rating: string (nullable = true)
 |-- duration: string (nullable = true)
 |-- listed_in: string (nullable = true)
 |-- description: string (nullable = true)



In [5]:
from pyspark.sql import functions as F

null_counts = raw.select([
    F.sum(
        F.when(F.col(c).isNull(), 1).otherwise(0)
    ).alias(c)
    for c in raw.columns
])

print("Null Values in Each Column:")
null_counts.show(truncate=False)

Null Values in Each Column:
+-------+----+-----+--------+----+-------+----------+------------+------+--------+---------+-----------+
|show_id|type|title|director|cast|country|date_added|release_year|rating|duration|listed_in|description|
+-------+----+-----+--------+----+-------+----------+------------+------+--------+---------+-----------+
|0      |1   |2    |2636    |826 |832    |13        |2           |6     |5       |3        |3          |
+-------+----+-----+--------+----+-------+----------+------------+------+--------+---------+-----------+



In [6]:
clean = (
    raw

    # 1. Drop rows with no title and remove duplicate IDs
    .dropna(subset=["show_id", "title"])
    .dropDuplicates(["show_id"])

    # 2. Fill missing categorical values
    .na.fill({
        "country": "Unknown",
        "director": "Unknown",
        "cast": "Unknown",
        "rating": "Not Rated",
        "listed_in": "Unknown"
    })

    # 3. Fix data types
    .withColumn(
        "release_year",
        F.col("release_year").cast("int")
    )
    .withColumn(
        "date_added",
        F.to_date(
            F.trim(F.col("date_added")),
            "MMMM d, yyyy"
        )
    )

    # 4. Derive helpful columns
    .withColumn(
        "content_age",
        F.year(F.current_date()) - F.col("release_year")
    )
    .withColumn(
        "primary_country",
        F.trim(
            F.split(F.col("country"), ",").getItem(0)
        )
    )

    # 5. Explode multi-value genres into one row per genre
    .withColumn(
        "genre",
        F.explode(
            F.split(F.col("listed_in"), ",\\s*")
        )
    )
)

print("Transformation chain defined successfully.")

Transformation chain defined successfully.


In [7]:
print("Raw rows:", raw.count(), " Clean rows:", clean.count())

clean.select(
    "title",
    "type",
    "primary_country",
    "release_year",
    "content_age",
    "genre"
).show(10, truncate=False)

Raw rows: 8809  Clean rows: 19304
+--------------------+-------+---------------+------------+-----------+--------------------+
|title               |type   |primary_country|release_year|content_age|genre               |
+--------------------+-------+---------------+------------+-----------+--------------------+
|Dick Johnson Is Dead|Movie  |United States  |2020        |6          |Documentaries       |
|The Starling        |Movie  |United States  |2021        |5          |Comedies            |
|The Starling        |Movie  |United States  |2021        |5          |Dramas              |
|On the Verge        |TV Show|France         |2021        |5          |TV Comedies         |
|On the Verge        |TV Show|France         |2021        |5          |TV Dramas           |
|Stowaway            |Movie  |Germany        |2021        |5          |Dramas              |
|Stowaway            |Movie  |Germany        |2021        |5          |International Movies|
|Stowaway            |Movie  |German

In [10]:
clean.printSchema()

root
 |-- show_id: string (nullable = true)
 |-- type: string (nullable = true)
 |-- title: string (nullable = true)
 |-- director: string (nullable = false)
 |-- cast: string (nullable = false)
 |-- country: string (nullable = false)
 |-- date_added: date (nullable = true)
 |-- release_year: integer (nullable = true)
 |-- rating: string (nullable = false)
 |-- duration: string (nullable = true)
 |-- listed_in: string (nullable = false)
 |-- description: string (nullable = true)
 |-- content_age: integer (nullable = true)
 |-- primary_country: string (nullable = true)
 |-- genre: string (nullable = false)



In [12]:
row_count = clean.count()
column_count = len(clean.columns)

print("Dataset Dimensions")
print("-------------------")
print("Total Rows:", row_count)
print("Total Columns:", column_count)

Dataset Dimensions
-------------------
Total Rows: 19304
Total Columns: 15


In [23]:
spark.stop()